# Standalone RoBERTa Classifier Baseline
This notebook implements the Standalone RoBERTa-base classifier model baseline using PyTorch.

### Setup Instructions:
1. Upload the dataset `fake_job_postings.csv` to the left files panel in Google Colab (drag-and-drop), or place it in the same directory as this notebook.
2. Set Colab runtime to **GPU** (T4) or **TPU**.
3. Run all cells sequentially.

In [ ]:
!pip install transformers

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

csv_path = "fake_job_postings.csv"

if not os.path.exists(csv_path):
    try:
        from google.colab import files
        print("Dataset 'fake_job_postings.csv' not found. Please upload it:")
        uploaded = files.upload()
    except ImportError:
        print(f"Local file '{csv_path}' not found. Please place it in the same directory.")
else:
    print(f"Dataset found at '{csv_path}'. Skipping upload prompt.")

## Preprocessing & Dataset

In [ ]:
df = pd.read_csv(csv_path)
df.fillna(" ", inplace=True)
df['combined_text'] = df['title'] + " " + df['company_profile'] + " " + df['description'] + " " + df['requirements'] + " " + df['benefits']

train_df, test_df = train_test_split(df, test_size=0.20, stratify=df['fraudulent'], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.125, stratify=train_df['fraudulent'], random_state=42)

class BERTDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.labels = df['fraudulent'].values
        self.texts = df['combined_text'].values
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

## Define Standalone RoBERTa Classifier

In [ ]:
class RoBERTaStandalone(nn.Module):
    def __init__(self, model_name="roberta-base", output_dim=1, dropout=0.3, freeze_bert=True):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(model_name)
        transformer_hidden_dim = self.transformer.config.hidden_size
        
        if freeze_bert:
            for param in self.transformer.parameters():
                param.requires_grad = False
                
        self.fc = nn.Linear(transformer_hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input_ids, attention_mask):
        with torch.set_grad_enabled(self.transformer.training and any(p.requires_grad for p in self.transformer.parameters())):
            outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
            pooled = outputs.last_hidden_state[:, 0, :] # Extract classification representations
            
        logits = self.fc(self.dropout(pooled))
        return logits.squeeze(1)

## Train RoBERTa model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
train_dataset = BERTDataset(train_df, tokenizer)
val_dataset = BERTDataset(val_df, tokenizer)
test_dataset = BERTDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model = RoBERTaStandalone(freeze_bert=True).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

pos_weight = torch.tensor([2.0]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

best_f1 = 0.0
epochs = 3

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    # Validate
    model.eval()
    val_probs = []
    val_labels = []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label']
            
            outputs = torch.sigmoid(model(input_ids, attention_mask))
            val_probs.extend(outputs.cpu().numpy())
            val_labels.extend(labels.numpy())
            
    val_labels_int = [int(l) for l in val_labels]
    val_preds = (np.array(val_probs) >= 0.5).astype(int)
    report = classification_report(val_labels_int, val_preds, output_dict=True, zero_division=0)
    val_f1 = report.get('1', report.get('1.0', report.get(1, {}))).get('f1-score', 0.0)
    
    print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f}")
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "roberta_standalone_best.pt")
        print("  Saved new best model checkpoint!")

## Evaluate on Test Set

In [ ]:
model.load_state_dict(torch.load("roberta_standalone_best.pt"))
model.eval()

test_probs = []
test_labels = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label']
        
        outputs = torch.sigmoid(model(input_ids, attention_mask))
        test_probs.extend(outputs.cpu().numpy())
        test_labels.extend(labels.numpy())

test_preds = (np.array(test_probs) >= 0.5).astype(int)
print("Standalone RoBERTa Classification Report:")
print(classification_report(test_labels, test_preds))

# Plot Confusion Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(test_labels, test_preds), annot=True, fmt='d', cmap='Oranges')
plt.title('Standalone RoBERTa Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()